# Email Classifier — evaluation + QLoRA fine-tuning on a free Colab GPU

**Before you start:** menu **Runtime → Change runtime type → T4 GPU → Save**.

Then run the cells top to bottom (**Runtime → Run all**). Step 4 asks you to upload `personal_test.csv`.

What this notebook does, step by step:
1. Checks the GPU
2. Downloads your project from GitHub
3. Installs packages
4. Uploads your labeled emails (`personal_test.csv`)
5. Runs the unit tests
6. **Evaluates the base model** (phishing benchmark + your inbox)
7. **Fine-tunes with QLoRA** (keeps 25% of your emails aside as a held-out test set)
8. **Compares base vs fine-tuned** on the held-out emails
9. Zips all results for you to download

Total time on a T4: roughly 60–90 minutes.

In [ ]:
# 1. Check the GPU (should say Tesla T4 or similar)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Download the project from GitHub
import os, getpass, subprocess
REPO = "RakshithR420/Email_Classifier"
if not os.path.exists("Email_Classifier"):
    r = subprocess.run(["git", "clone", f"https://github.com/{REPO}.git"], capture_output=True, text=True)
    if r.returncode != 0:
        # Private repo: paste a GitHub personal access token (it is not saved anywhere).
        token = getpass.getpass("Repo is private. Paste a GitHub token (read access): ")
        subprocess.run(["git", "clone", f"https://{token}@github.com/{REPO}.git"], check=True)
%cd /content/Email_Classifier
!git log --oneline -3

In [ ]:
# 3. Install packages (Colab already has PyTorch with GPU support)
!pip install -q -U bitsandbytes peft accelerate "transformers>=4.51" pyyaml scikit-learn tqdm pytest

In [ ]:
# 4. Upload personal_test.csv (from Desktop\\Email_Classifier\\eval\\data on your PC)
from google.colab import files
import shutil
uploaded = files.upload()
name = next(iter(uploaded))
shutil.move(name, "eval/data/personal_test.csv")
import pandas as pd
df = pd.read_csv("eval/data/personal_test.csv")
print(len(df), "labeled emails"); print(df["true_category"].value_counts())

In [ ]:
# 5. Unit tests (should say 24 passed)
!python -m pytest -q

In [ ]:
# 6. Base model: full evaluation (phishing benchmark + all your labeled emails)
!mkdir -p results
!python eval/evaluate.py 2>&1 | grep -v "it/s]" | tee results/1_base_full.txt

In [ ]:
# 7. QLoRA fine-tuning (creates eval/data/personal_holdout.csv = emails the model never trains on)
!python finetune/train_qlora.py --epochs 3 2>&1 | grep -v "it/s]" | tee results/2_training_log.txt

In [ ]:
# 8a. Base model on the held-out emails (fair baseline)
!python eval/evaluate.py --only personal --personal-file eval/data/personal_holdout.csv 2>&1 | grep -v "it/s]" | tee results/3_base_holdout.txt

In [ ]:
# 8b. Switch to the fine-tuned adapter and evaluate the SAME held-out emails + phishing benchmark
import yaml
cfg = yaml.safe_load(open("config/settings.yaml"))
cfg["model"]["adapter_path"] = "finetune/adapters/qwen3-8b-email"
yaml.safe_dump(cfg, open("config/settings.yaml", "w"), sort_keys=False)
!python eval/evaluate.py --personal-file eval/data/personal_holdout.csv 2>&1 | grep -v "it/s]" | tee results/4_finetuned_holdout.txt

In [ ]:
# 9. Summary + download everything
import re, glob
def acc(path, title):
    txt = open(path).read()
    blocks = txt.split("=== ")
    for b in blocks:
        if b.startswith(title):
            m = re.search(r"accuracy\s+([0-9.]+)\s+(\d+)", b)
            return f"{float(m.group(1)):.0%} on {m.group(2)} emails" if m else "n/a"
    return "n/a"
print("Base model, phishing benchmark  :", acc("results/1_base_full.txt", "Phishing"))
print("Base model, all personal emails :", acc("results/1_base_full.txt", "Personal"))
print("Base model, held-out emails     :", acc("results/3_base_holdout.txt", "Personal"))
print("Fine-tuned, held-out emails     :", acc("results/4_finetuned_holdout.txt", "Personal"))
print("Fine-tuned, phishing benchmark  :", acc("results/4_finetuned_holdout.txt", "Phishing"))

!cp eval/data/*predictions.csv results/ 2>/dev/null; cp finetune/adapters/qwen3-8b-email/training_info.json results/ 2>/dev/null
!zip -qr results.zip results
files.download("results.zip")

**Next:** put the downloaded `results.zip` in your `Desktop\\Email_Classifier` folder and tell Claude — it will add the numbers to `eval/results.md` and your resume.

*Optional:* to keep the fine-tuned adapter (~100–200 MB), run `!zip -qr adapter.zip finetune/adapters` and `files.download("adapter.zip")`.